# Spam Classification with RNN & CNN

In [14]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

## 1. Load Dataset

In [15]:
data = []
with open("spam.txt", "r", encoding="utf-8") as f:
    for line in f:
        if "\t" in line:
            text, label = line.strip().split("\t")
            data.append((text, int(label)))

df = pd.DataFrame(data, columns=["text", "label"])
print("Dataset size:", len(df))
print(df.head())

Dataset size: 1547
                                                text  label
0  Go until jurong point, crazy.. Available only ...      0
1                      Ok lar... Joking wif u oni...      0
2  U dun say so early hor... U c already then say...      0
3  Nah I don't think he goes to usf, he lives aro...      0
4  Even my brother is not like to speak with me. ...      0


## 2. Preprocessing

In [16]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text

df["clean_text"] = df["text"].apply(clean_text)


with open("checkpoints/vocab.pkl", "rb") as f:
    vocab_data = pickle.load(f)

vocab = vocab_data["vocab"]
word2idx = vocab_data["word2idx"]
idx2word = vocab_data["idx2word"]

PAD_IDX = word2idx.get("<pad>", 0)
UNK_IDX = word2idx.get("<unk>", PAD_IDX)

def encode_sentence(sentence, max_len=50):
    tokens = sentence.split()
    ids = [word2idx.get(t, UNK_IDX) for t in tokens]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

MAX_LEN = 50
X = np.array([encode_sentence(s, MAX_LEN) for s in df["clean_text"]])
y = df["label"].values

## 3. Load Pretrained Embeddings

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, embedding_dim, svd_tensor):
        super().__init__()
        self.in_embeddings = nn.Embedding.from_pretrained(svd_tensor.clone().float(), freeze=False, padding_idx=PAD_IDX)
        self.out_embeddings = nn.Embedding(vocab_size, embedding_dim)
        nn.init.uniform_(self.out_embeddings.weight, -0.5/embedding_dim, 0.5/embedding_dim)

    def forward(self, center_words, pos_context_words, neg_context_words):
        c = self.in_embeddings(center_words)                     
        pos = self.out_embeddings(pos_context_words)            
        neg = self.out_embeddings(neg_context_words)             

        pos_score = torch.sum(c * pos, dim=1)                   
        pos_loss = torch.log(torch.sigmoid(pos_score) + 1e-10)

        neg_score = torch.bmm(neg, c.unsqueeze(2)).squeeze(2)    
        neg_loss = torch.log(torch.sigmoid(-neg_score) + 1e-10).sum(1)

        loss = -(pos_loss + neg_loss).mean()
        return loss

def find_last_checkpoint(directory, prefix="checkpoint_"):
    if not os.path.isdir(directory):
        return None
    checkpoints = [f for f in os.listdir(directory) if f.endswith(".pt") and prefix in f]
    if not checkpoints:
        return None
    epochs = []
    for f in checkpoints:
        try:
            num = int(f.replace(prefix, "").replace(".pt", ""))
            epochs.append((num, f))
        except:
            continue
    if not epochs:
        return None
    latest = max(epochs, key=lambda x: x[0])[1]
    return os.path.join(directory, latest)

ckpt_path = find_last_checkpoint("checkpoints")
if ckpt_path is None:
    raise FileNotFoundError("No checkpoint found in checkpoints/")

checkpoint = torch.load(ckpt_path, map_location=device)
embedding_dim = checkpoint["model_state_dict"]["in_embeddings.weight"].shape[1]
vocab_size = checkpoint["model_state_dict"]["in_embeddings.weight"].shape[0]

sg_model = SkipGramNegSampling(vocab_size, embedding_dim, torch.rand(vocab_size, embedding_dim)).to(device)
sg_model.load_state_dict(checkpoint["model_state_dict"])

pretrained_weights = sg_model.in_embeddings.weight.detach().float().cpu()
print("Loaded pretrained embeddings:", pretrained_weights.shape)

Loaded pretrained embeddings: torch.Size([34154, 300])


## 4. Dataset & DataLoader

In [18]:
class SMSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## 5. RNN Classifier

In [19]:
class RNNClassifier(nn.Module):
    def __init__(self, pretrained_weights, hidden_dim=128, num_layers=1, num_classes=2):
        super(RNNClassifier, self).__init__()
        vocab_size, embedding_dim = pretrained_weights.shape
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=False, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        out, (hn, cn) = self.rnn(x)
        if self.rnn.bidirectional:
            h_forward = hn[-2, :, :]
            h_backward = hn[-1, :, :]
            h = torch.cat((h_forward, h_backward), dim=1)
        else:
            h = hn[-1]
        return self.fc(h)

## 6. CNN Classifier

In [20]:
class CNNClassifier(nn.Module):
    def __init__(self, pretrained_weights, num_classes=2, num_filters=100, filter_sizes=(3,4,5)):
        super(CNNClassifier, self).__init__()
        vocab_size, embedding_dim = pretrained_weights.shape
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=False, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, embedding_dim)) for fs in filter_sizes
        ])
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x)                   
        x = x.unsqueeze(1)                     
        conv_outs = []
        for conv in self.convs:
            c = torch.relu(conv(x))             
            c = c.squeeze(3)                    
            c = torch.max(c, dim=2)[0]          
            conv_outs.append(c)
        x = torch.cat(conv_outs, dim=1)
        x = self.dropout(x)
        return self.fc(x)

## 7. Training & Evaluation Utils

In [24]:
def train_eval(model_class, pretrained_weights, X, y, folds=6, epochs=5, batch_size=64, save_path=None):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    metrics = {"acc": [], "prec": [], "rec": [], "f1": []}
    best_model = None
    best_f1 = 0

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        print(f"\n===== Fold {fold+1} =====")
        train_ds = SMSDataset(X[train_idx], y[train_idx])
        test_ds = SMSDataset(X[test_idx], y[test_idx])
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=batch_size)

        model = model_class(pretrained_weights).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        
        for epoch in range(epochs):
            total_loss = 0
            model.train()
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                optimizer.zero_grad()
                preds = model(Xb)
                loss = criterion(preds, yb)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

       
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb = Xb.to(device)
                preds = model(Xb).argmax(dim=1).cpu().numpy()
                y_true.extend(yb.numpy())
                y_pred.extend(preds)

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        metrics["acc"].append(acc)
        metrics["prec"].append(prec)
        metrics["rec"].append(rec)
        metrics["f1"].append(f1)

        print(f"Fold {fold+1} → Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

        
        if f1 > best_f1:
            best_f1 = f1
            best_model = model.state_dict()

    for k in metrics:
        print(f"{k.upper()}: {np.mean(metrics[k]):.4f} ± {np.std(metrics[k]):.4f}")

    if save_path and best_model:
        torch.save(best_model, save_path)
        print(f" Best model saved to {save_path}")

    return metrics


## 8. Run Experiments

In [26]:
print("\n===== CNN Model Results =====")
rnn_metrics = train_eval(RNNClassifier, pretrained_weights, X, y, save_path="checkpoints/rnn_final.pt")

print("\n===== RNN Model Results =====")
cnn_metrics = train_eval(CNNClassifier, pretrained_weights, X, y, save_path="checkpoints/cnn_final.pt")



===== CNN Model Results =====

===== Fold 1 =====
Epoch 1, Loss: 0.4946
Epoch 2, Loss: 0.1393
Epoch 3, Loss: 0.0562
Epoch 4, Loss: 0.0245
Epoch 5, Loss: 0.0126
Fold 1 → Acc: 0.9922, Prec: 1.0000, Rec: 0.9832, F1: 0.9915

===== Fold 2 =====
Epoch 1, Loss: 0.4848
Epoch 2, Loss: 0.1029
Epoch 3, Loss: 0.0333
Epoch 4, Loss: 0.0195
Epoch 5, Loss: 0.0115
Fold 2 → Acc: 0.9690, Prec: 0.9920, Rec: 0.9466, F1: 0.9688

===== Fold 3 =====
Epoch 1, Loss: 0.4912
Epoch 2, Loss: 0.1239
Epoch 3, Loss: 0.0366
Epoch 4, Loss: 0.0154
Epoch 5, Loss: 0.0077
Fold 3 → Acc: 0.9806, Prec: 0.9912, Rec: 0.9655, F1: 0.9782

===== Fold 4 =====
Epoch 1, Loss: 0.4700
Epoch 2, Loss: 0.0904
Epoch 3, Loss: 0.0285
Epoch 4, Loss: 0.0210
Epoch 5, Loss: 0.0102
Fold 4 → Acc: 0.9729, Prec: 0.9829, Rec: 0.9583, F1: 0.9705

===== Fold 5 =====
Epoch 1, Loss: 0.5014
Epoch 2, Loss: 0.1327
Epoch 3, Loss: 0.0444
Epoch 4, Loss: 0.0274
Epoch 5, Loss: 0.0124
Fold 5 → Acc: 0.9806, Prec: 1.0000, Rec: 0.9621, F1: 0.9807

===== Fold 6 =====

In [27]:
def predict(sentence, model_type="cnn"):
    x = torch.tensor([encode_sentence(sentence)], dtype=torch.long).to(device)

    if model_type == "cnn":
        model = CNNClassifier(pretrained_weights)
        model.load_state_dict(torch.load("checkpoints/cnn_final.pt", map_location=device))
    else:
        model = RNNClassifier(pretrained_weights)
        model.load_state_dict(torch.load("checkpoints/rnn_final.pt", map_location=device))

    model = model.to(device)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        pred_label = torch.argmax(probs, dim=1).item()
        confidence = probs[0, pred_label].item()

    label_str = "SPAM" if pred_label == 1 else "NOT SPAM"
    print(f"\n Input: {sentence}")
    print(f" Prediction: {label_str}  (Confidence: {confidence:.4f})")


In [28]:
predict("Congratulations! You have won a free iPhone. Click the link to claim.", "cnn")
predict("Congratulations! You have won a free iPhone. Click the link to claim.", "rnn")

predict("Okay, see you at the gym later.", "cnn")
predict("Okay, see you at the gym later.", "rnn")


 Input: Congratulations! You have won a free iPhone. Click the link to claim.
 Prediction: SPAM  (Confidence: 1.0000)

 Input: Congratulations! You have won a free iPhone. Click the link to claim.
 Prediction: SPAM  (Confidence: 0.9986)

 Input: Okay, see you at the gym later.
 Prediction: NOT SPAM  (Confidence: 0.9992)

 Input: Okay, see you at the gym later.
 Prediction: NOT SPAM  (Confidence: 0.9962)
